# Tech Challenge Fase 3 — Análise Exploratória de Dados (EDA)

## Objetivo

Realizar a análise exploratória da Gold individual de modelagem:

`workspace.alfabetizacao_gold.base_modelagem_aluno`

A análise busca responder:

1. Como o target `alfabetizado` se distribui em 2023 e 2024?
2. Quais diferenças aparecem entre redes e UFs?
3. Como as variáveis socioeconômicas se relacionam com o target?
4. Quais variáveis devem seguir para a pipeline de Machine Learning?

### Estratégia temporal

- **2023** → desenvolvimento, treinamento e cross-validation;
- **2024** → teste temporal final *out-of-time*.

O `id_aluno` é tratado apenas como identificador técnico e não como feature preditora.


## 1. Configurações


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

import matplotlib.pyplot as plt

CATALOG = "workspace"
GOLD_SCHEMA = "alfabetizacao_gold"

MODEL_TABLE = f"{CATALOG}.{GOLD_SCHEMA}.base_modelagem_aluno"

df = spark.table(MODEL_TABLE)

print("Tabela:", MODEL_TABLE)
print("Registros:", df.count())
print("Colunas:", len(df.columns))
print(df.columns)


## 2. Visão geral da base

Validamos volume, período, grupos metodológicos, municípios, UFs e distribuição do target.


In [ ]:
display(
    df
    .groupBy("ano", "grupo_modelagem")
    .agg(
        F.count("*").alias("registros"),
        F.countDistinct("id_aluno").alias("alunos_distintos"),
        F.countDistinct("id_municipio").alias("municipios"),
        F.countDistinct("sigla_uf").alias("ufs"),
        F.round(F.avg("target_alfabetizado") * 100, 2)
            .alias("percentual_alfabetizados"),
    )
    .orderBy("ano")
)


## 3. Distribuição do target

O target está próximo de 50/50, então não há indicação inicial de necessidade de oversampling ou undersampling.


In [ ]:
target_ano = (
    df
    .groupBy("ano", "target_alfabetizado")
    .agg(F.count("*").alias("registros"))
    .withColumn(
        "classe",
        F.when(F.col("target_alfabetizado") == 1, "Alfabetizado")
         .otherwise("Não alfabetizado")
    )
)

display(target_ano.orderBy("ano", "target_alfabetizado"))


In [ ]:
pdf_target = (
    target_ano
    .select("ano", "classe", "registros")
    .toPandas()
)

pivot_target = (
    pdf_target
    .pivot(index="ano", columns="classe", values="registros")
    .fillna(0)
)

ax = pivot_target.plot(kind="bar", figsize=(8, 5))
ax.set_title("Distribuição do target por ano")
ax.set_xlabel("Ano")
ax.set_ylabel("Quantidade de alunos")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 4. Alfabetização por rede

A rede de ensino é uma variável educacional simples e interpretável.


In [ ]:
rede_resumo = (
    df
    .groupBy("ano", "rede")
    .agg(
        F.count("*").alias("alunos"),
        F.round(F.avg("target_alfabetizado") * 100, 2)
            .alias("taxa_alfabetizacao"),
    )
    .orderBy("ano", F.desc("alunos"))
)

display(rede_resumo)


In [ ]:
pdf_rede = rede_resumo.toPandas()

for ano in sorted(pdf_rede["ano"].unique()):
    dados = (
        pdf_rede[pdf_rede["ano"] == ano]
        .sort_values("taxa_alfabetizacao", ascending=False)
    )

    ax = dados.plot(
        x="rede",
        y="taxa_alfabetizacao",
        kind="bar",
        legend=False,
        figsize=(7, 4),
    )
    ax.set_title(f"Taxa de alfabetização por rede — {ano}")
    ax.set_xlabel("Rede")
    ax.set_ylabel("Taxa de alfabetização (%)")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()


## 5. Alfabetização por UF

A análise por UF permite observar heterogeneidade territorial sem usar diretamente o município como feature categórica de alta cardinalidade.


In [ ]:
uf_resumo = (
    df
    .groupBy("ano", "sigla_uf")
    .agg(
        F.count("*").alias("alunos"),
        F.countDistinct("id_municipio").alias("municipios"),
        F.round(F.avg("target_alfabetizado") * 100, 2)
            .alias("taxa_alfabetizacao"),
    )
    .orderBy("ano", F.desc("taxa_alfabetizacao"))
)

display(uf_resumo)


In [ ]:
pdf_uf_2024 = (
    uf_resumo
    .filter(F.col("ano") == 2024)
    .orderBy(F.desc("taxa_alfabetizacao"))
    .toPandas()
)

ax = pdf_uf_2024.plot(
    x="sigla_uf",
    y="taxa_alfabetizacao",
    kind="bar",
    legend=False,
    figsize=(12, 5),
)
ax.set_title("Taxa de alfabetização por UF — 2024")
ax.set_xlabel("UF")
ax.set_ylabel("Taxa de alfabetização (%)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 6. Municípios com maiores e menores taxas

Para reduzir conclusões instáveis, o ranking considera apenas municípios com pelo menos **100 alunos** no respectivo ano.

O `id_municipio` será usado para análise e rastreabilidade, não como feature inicial.


In [ ]:
municipio_resumo = (
    df
    .groupBy("ano", "id_municipio", "nome_municipio", "sigla_uf")
    .agg(
        F.count("*").alias("alunos"),
        F.round(F.avg("target_alfabetizado") * 100, 2)
            .alias("taxa_alfabetizacao"),
    )
    .filter(F.col("alunos") >= 100)
)

print("10 maiores taxas em 2024:")
display(
    municipio_resumo
    .filter(F.col("ano") == 2024)
    .orderBy(F.desc("taxa_alfabetizacao"), F.desc("alunos"))
    .limit(10)
)

print("10 menores taxas em 2024:")
display(
    municipio_resumo
    .filter(F.col("ano") == 2024)
    .orderBy(F.asc("taxa_alfabetizacao"), F.desc("alunos"))
    .limit(10)
)


## 7. Completude das variáveis

A completude é validada novamente antes da modelagem.


In [ ]:
variaveis_validacao = [
    "rede",
    "sigla_uf",
    "populacao",
    "quantidade_vinculos_ativos",
    "quantidade_vinculos_clt",
    "quantidade_vinculos_estatutarios",
    "vinculos_ativos_por_1000_habitantes",
    "target_alfabetizado",
]

expressoes_nulos = [
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(f"{c}_nulos")
    for c in variaveis_validacao
]

display(
    df
    .groupBy("ano")
    .agg(*expressoes_nulos)
    .orderBy("ano")
)


## 8. Estatísticas descritivas das variáveis numéricas

As estatísticas ajudam a identificar assimetria, amplitude e diferenças entre desenvolvimento e teste temporal.


In [ ]:
numeric_cols = [
    "populacao",
    "quantidade_vinculos_ativos",
    "quantidade_vinculos_clt",
    "quantidade_vinculos_estatutarios",
    "vinculos_ativos_por_1000_habitantes",
]

for ano in [2023, 2024]:
    print(f"Estatísticas descritivas — {ano}")
    display(
        df
        .filter(F.col("ano") == ano)
        .select(*numeric_cols)
        .summary("count", "mean", "stddev", "min", "25%", "50%", "75%", "max")
    )


## 9. Variáveis socioeconômicas por classe do target

A comparação é descritiva: diferenças entre médias não implicam causalidade.


In [ ]:
display(
    df
    .groupBy("ano", "target_alfabetizado")
    .agg(
        F.count("*").alias("alunos"),
        F.round(F.avg("populacao"), 2).alias("populacao_media"),
        F.round(F.avg("quantidade_vinculos_ativos"), 2)
            .alias("vinculos_ativos_media"),
        F.round(F.avg("quantidade_vinculos_clt"), 2)
            .alias("vinculos_clt_media"),
        F.round(F.avg("quantidade_vinculos_estatutarios"), 2)
            .alias("vinculos_estatutarios_media"),
        F.round(F.avg("vinculos_ativos_por_1000_habitantes"), 2)
            .alias("vinculos_por_1000_media"),
    )
    .orderBy("ano", "target_alfabetizado")
)


## 10. População e alfabetização por decis

A população é dividida em decis dentro de cada ano para reduzir o efeito de valores extremos.


In [ ]:
w_pop = Window.partitionBy("ano").orderBy(F.col("populacao"))

df_decis_pop = df.withColumn(
    "decil_populacao",
    F.ntile(10).over(w_pop),
)

pop_decis_resumo = (
    df_decis_pop
    .groupBy("ano", "decil_populacao")
    .agg(
        F.count("*").alias("alunos"),
        F.round(F.avg("populacao"), 0).alias("populacao_media"),
        F.round(F.avg("target_alfabetizado") * 100, 2)
            .alias("taxa_alfabetizacao"),
    )
    .orderBy("ano", "decil_populacao")
)

display(pop_decis_resumo)


In [ ]:
pdf_pop_decis = pop_decis_resumo.toPandas()

for ano in sorted(pdf_pop_decis["ano"].unique()):
    dados = pdf_pop_decis[pdf_pop_decis["ano"] == ano]
    ax = dados.plot(
        x="decil_populacao",
        y="taxa_alfabetizacao",
        kind="line",
        marker="o",
        legend=False,
        figsize=(8, 4),
    )
    ax.set_title(f"Taxa de alfabetização por decil de população — {ano}")
    ax.set_xlabel("Decil de população")
    ax.set_ylabel("Taxa de alfabetização (%)")
    ax.set_xticks(range(1, 11))
    plt.tight_layout()
    plt.show()


## 11. Mercado formal e alfabetização por decis

O indicador `vinculos_ativos_por_1000_habitantes` normaliza o tamanho do mercado formal pelo porte populacional.


In [ ]:
w_vinc = (
    Window
    .partitionBy("ano")
    .orderBy(F.col("vinculos_ativos_por_1000_habitantes"))
)

df_decis_vinc = df.withColumn(
    "decil_vinculos",
    F.ntile(10).over(w_vinc),
)

vinc_decis_resumo = (
    df_decis_vinc
    .groupBy("ano", "decil_vinculos")
    .agg(
        F.count("*").alias("alunos"),
        F.round(F.avg("vinculos_ativos_por_1000_habitantes"), 2)
            .alias("vinculos_por_1000_media"),
        F.round(F.avg("target_alfabetizado") * 100, 2)
            .alias("taxa_alfabetizacao"),
    )
    .orderBy("ano", "decil_vinculos")
)

display(vinc_decis_resumo)


In [ ]:
pdf_vinc_decis = vinc_decis_resumo.toPandas()

for ano in sorted(pdf_vinc_decis["ano"].unique()):
    dados = pdf_vinc_decis[pdf_vinc_decis["ano"] == ano]
    ax = dados.plot(
        x="decil_vinculos",
        y="taxa_alfabetizacao",
        kind="line",
        marker="o",
        legend=False,
        figsize=(8, 4),
    )
    ax.set_title(f"Taxa de alfabetização por decil de vínculos formais — {ano}")
    ax.set_xlabel("Decil de vínculos ativos por 1.000 habitantes")
    ax.set_ylabel("Taxa de alfabetização (%)")
    ax.set_xticks(range(1, 11))
    plt.tight_layout()
    plt.show()


## 12. Correlação das variáveis numéricas com o target

Como o target é binário (`0/1`), a correlação de Pearson com uma variável contínua equivale à correlação ponto-bisserial.


In [ ]:
corr_rows = []

for ano in [2023, 2024]:
    df_ano = df.filter(F.col("ano") == ano)

    for coluna in numeric_cols:
        valor = df_ano.stat.corr(coluna, "target_alfabetizado")
        corr_rows.append(
            (ano, coluna, float(valor) if valor is not None else None)
        )

corr_df = spark.createDataFrame(
    corr_rows,
    ["ano", "variavel", "correlacao_target"],
)

display(
    corr_df
    .orderBy("ano", F.desc(F.abs(F.col("correlacao_target"))))
)


## 13. Redundância entre variáveis socioeconômicas

A matriz abaixo ajuda a identificar multicolinearidade entre as medidas de porte e mercado formal.


In [ ]:
corr_pairs = []

for i, col_a in enumerate(numeric_cols):
    for col_b in numeric_cols[i + 1:]:
        valor = df.stat.corr(col_a, col_b)
        corr_pairs.append(
            (col_a, col_b, float(valor) if valor is not None else None)
        )

corr_pairs_df = spark.createDataFrame(
    corr_pairs,
    ["variavel_1", "variavel_2", "correlacao"],
)

display(
    corr_pairs_df
    .orderBy(F.desc(F.abs(F.col("correlacao"))))
)


## 14. Comparação temporal 2023 × 2024

A comparação entre desenvolvimento e teste ajuda a identificar mudanças de distribuição antes da modelagem.


In [ ]:
display(
    df
    .groupBy("ano")
    .agg(
        F.count("*").alias("alunos"),
        F.countDistinct("id_municipio").alias("municipios"),
        F.countDistinct("sigla_uf").alias("ufs"),
        F.round(F.avg("target_alfabetizado") * 100, 2)
            .alias("taxa_alfabetizacao"),
        F.round(F.avg("populacao"), 2).alias("populacao_media"),
        F.round(F.avg("quantidade_vinculos_ativos"), 2)
            .alias("vinculos_ativos_media"),
        F.round(F.avg("vinculos_ativos_por_1000_habitantes"), 2)
            .alias("vinculos_por_1000_media"),
    )
    .orderBy("ano")
)


## 15. Cardinalidade das variáveis categóricas

A cardinalidade orienta as decisões de encoding.


In [ ]:
categorical_cols = ["rede", "sigla_uf", "id_municipio"]

cardinalidade = [
    (coluna, df.select(coluna).distinct().count())
    for coluna in categorical_cols
]

display(
    spark.createDataFrame(
        cardinalidade,
        ["variavel", "cardinalidade"],
    )
)


## 16. Decisão preliminar de features

A pipeline inicial será mantida simples e interpretável.

### Features categóricas candidatas

- `rede`
- `sigla_uf`

### Features numéricas candidatas

- `populacao`
- `quantidade_vinculos_ativos`
- `quantidade_vinculos_clt`
- `quantidade_vinculos_estatutarios`
- `vinculos_ativos_por_1000_habitantes`

### Variáveis que não entram no modelo inicial

- `id_aluno`: identificador técnico;
- `id_municipio`: alta cardinalidade e risco de memorização territorial;
- `nome_municipio`: representação textual redundante;
- `ano`: constante no conjunto de desenvolvimento de 2023;
- `ano_rais`: metadado temporal;
- `presenca`: informação do momento da avaliação e potencialmente indisponível em uma previsão anterior à prova;
- `proficiencia`: leakage direto do target;
- `serie`: variância zero;
- `peso_aluno`: peso amostral;
- `id_escola`: identificador de alta cardinalidade;
- `caderno` e `preenchimento_caderno`: variáveis operacionais.

### Multicolinearidade

As medidas absolutas de vínculos podem apresentar correlação elevada entre si e com a população.

No notebook de modelagem, compararemos uma versão completa com uma versão mais enxuta das features socioeconômicas.


## Conclusão

A EDA foi estruturada para apoiar diretamente as decisões de modelagem.

Os pontos que devem ser registrados após a execução são:

1. balanceamento do target;
2. diferenças entre redes e UFs;
3. comportamento territorial;
4. relação entre contexto socioeconômico e alfabetização;
5. estabilidade entre 2023 e 2024;
6. correlação e redundância entre features;
7. seleção final das variáveis.

### Próxima etapa

O próximo notebook será dedicado ao treinamento supervisionado, com:

- preprocessing integrado;
- encoding das categóricas;
- transformação das numéricas;
- baseline;
- comparação de modelos;
- cross-validation em 2023;
- avaliação final exclusivamente em 2024.
